| Task ID | Task Name | Type | Category | Description |
|---------|-----------|------|----------|-------------|
| 1 | Dosage Sensitivity Classification | Binary | Gene Property Prediction | Distinguishes between dosage-sensitive and dosage-insensitive transcription factors |

In [1]:
import os
import pandas as pd
import pickle
os.chdir(os.path.abspath(".."))
from utils.Gene_level_task_utils import evaluate_embeddings

# Read the dosage sensitivity data
with open('Dataset/Gene_level_task/dosage_sensitivity_TFs.pickle', 'rb') as f:
    dosage_data = pickle.load(f)

if os.path.exists("Dataset/Gene_level_task/dosage_sensitivity_TFs_data.pkl"):
    with open("Dataset/Gene_level_task/dosage_sensitivity_TFs_data.pkl", "rb") as f:
        converted_data = pickle.load(f)
    print("load dosage_sensitivity_TFs_data.pkl")
else:
    import mygene
    # Initialize the mygene client
    mg = mygene.MyGeneInfo()
    # Extract all Ensembl IDs from both lists
    ds_tfs = dosage_data['Dosage-sensitive TFs']
    di_tfs = dosage_data['Dosage-insensitive TFs']

    # Combine both lists for batch query
    all_ids = ds_tfs + di_tfs

    # Query mygene for gene info
    gene_info = mg.querymany(all_ids, scopes='ensembl.gene', fields=['symbol', 'name'], species='human')

    # Create a dictionary mapping Ensembl IDs to gene symbols
    id_to_symbol = {}
    for info in gene_info:
        if 'symbol' in info:
            id_to_symbol[info['query']] = info['symbol']

    # Convert lists to gene symbols
    ds_symbols = [id_to_symbol.get(ensembl_id, ensembl_id) for ensembl_id in ds_tfs]
    di_symbols = [id_to_symbol.get(ensembl_id, ensembl_id) for ensembl_id in di_tfs]

    # Create new dictionary with gene symbols
    converted_data = {
        'Dosage-sensitive TFs': ds_symbols,
        'Dosage-insensitive TFs': di_symbols
    }
    with open("Dataset/Gene_level_task/dosage_sensitivity_TFs_data.pkl", "wb") as f:
        pickle.dump(converted_data, f)
    print("save dosage_sensitivity_TFs_data.pkl")

methods = ["SCGPT-HUMAN","SCGPT-PANCANCER","GF-6L30M","GF-20L95M","GF-12L95M","GF-12L30M", "GF-12L95MCANCER","GENE2VEC","TCGA-EMBEDDING","FROGS-ARCHS4","MUT2VEC","GENEPT-MODEL3","GENEPT-ADA","BIOCONCEPTVEC-SKIP-GRAM","BIOCONCEPTVEC-FASTTEXT","BIOCONCEPTVEC-GLOVE","BIOCONCEPTVEC-CBOW","CoxFormer"]
file_paths = {method: f'Embeddings/{method}.pkl' for method in methods}

# Check if the CSV file exists
csv_file_path = "Result/Gene_level_task/Dosage_Sensitivity_Classification.csv"
if os.path.exists(csv_file_path):
    # Load the existing results
    results_df = pd.read_csv(csv_file_path, index_col=0)

    # Check if all methods are already in the results
    existing_methods = results_df['Method'].tolist()
    additional_methods = [method for method in methods if method not in existing_methods]

    if additional_methods:
        print(f"Running evaluation for additional methods: {additional_methods}")

        # Evaluate the additional methods
        new_results_df = evaluate_embeddings(
            data_dict=converted_data,
            positive_label='Dosage-sensitive TFs',
            negative_label='Dosage-insensitive TFs',
            methods=additional_methods,
            file_paths=file_paths
        )

        # Append new results to the existing DataFrame
        results_df = pd.concat([results_df, new_results_df], ignore_index=True)
    else:
        print("All methods are already evaluated. Skipping evaluation.")
else:
    # If the CSV file does not exist, evaluate all methods
    print("CSV file does not exist. Running evaluation for all methods.")
    results_df = evaluate_embeddings(
        data_dict=converted_data,
        positive_label='Dosage-sensitive TFs',
        negative_label='Dosage-insensitive TFs',
        methods=methods,
        file_paths=file_paths
    )

# Save the updated results to CSV
results_df.to_csv(csv_file_path)

load dosage_sensitivity_TFs_data.pkl
All methods are already evaluated. Skipping evaluation.
